# ASTRA-GUARD: Phase 5 — Mission Risk Engine & Explainable AI (XAI)
**AI-Powered Integrated Space Mission Safety & Intelligence**
Team OMEN · AITHON 2.0 · Track 11 Aerospace, Telemetry & SpaceTech

This notebook demonstrates:
1. **Noisy-OR Probabilistic Safety Hazard Fusion:** Integrating spacecraft health ($r_h$) and orbital debris collision risk ($r_d$) via $R_{safety} = 1 - (1 - r_h)(1 - r_d)$.
2. **Total Mission Risk Score ($R_{mission}$):** Fusing physical safety with Earth observation priority ($r_e$) via $100 \cdot (0.85 \cdot R_{safety} + 0.15 \cdot r_e)$.
3. **Hard Safety Overrides & Cross-Effects:** Enforcing $P_c \ge 10^{-4} \implies$ `CRITICAL`, persistent $r_h \ge 0.90 \implies \ge$ `WARNING`, and $r_h \ge 0.50 \implies$ EO task deferral.
4. **Attribution Aggregation:** Normalizing multi-module feature drivers to 100% of explained risk variance.
5. **Deterministic Aerospace Rationale & Operator Recommendations:** Rigorous flight-director templates and human-in-the-loop action codes.


In [1]:
import os
import sys

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, os.path.abspath('..'))

from ml.common.schemas import Attribution
from ml.risk_engine.engine import MissionRiskEngine

engine = MissionRiskEngine()
print('MissionRiskEngine initialized with config/risk_engine.yaml.')

## 1. Nominal Flight Operations (Scenario 1)
Subsystems nominal ($r_h = 0.04$), background debris encounter ($r_d = 0.02$), routine EO ($r_e = 0.10$).

In [2]:
snap1 = engine.evaluate(
    asset_id='SENTINEL-2A',
    health_input=0.04,
    debris_input=0.02,
    disaster_input=0.10,
    health_attributions=[Attribution(driver='eps_battery_temp_c', module='health', contribution=0.03, direction='increase')],
)
print(f'Score: {snap1.score:.1f} | Level: {snap1.level.value}')
print(f'Rationale: {snap1.rationale}')
print(f'Primary Recommendation: [{snap1.recommendations[0].action_code}] {snap1.recommendations[0].title}')

## 2. Spacecraft Subsystem Anomaly & EO Deferral (Scenario 2)
EPS thermal runaway anomaly ($r_h = 0.68$). Triggers cross-system deferral of secondary EO tasks.

In [3]:
snap2 = engine.evaluate(
    asset_id='SENTINEL-2A',
    health_input=0.68,
    debris_input=0.05,
    disaster_input=0.25,
    health_attributions=[
        Attribution(driver='eps_battery_temp_c', module='health', contribution=0.65, direction='increase'),
        Attribution(driver='eps_bus_voltage_v', module='health', contribution=0.25, direction='increase'),
    ],
    evidence={'anomaly_subsystem': 'EPS_BATTERY'},
)
print(f'Score: {snap2.score:.1f} | Level: {snap2.level.value} | Defer EO: {snap2.defer_eo_payloads}')
print(f'Top Driver: {snap2.drivers[0].driver} ({snap2.drivers[0].percent_of_risk}%)')
print(f'Recommendations count: {len(snap2.recommendations)}')
for r in snap2.recommendations:
    print(f' - [{r.action_code}] ({r.priority.upper()}) Action: {r.required_operator_action.value}')

## 3. Critical Debris Conjunction Hard Override (Scenario 3)
Close approach with Fengyun-1C debris ($P_c = 2.4\times 10^{-4} \ge 10^{-4}$). Hard override elevates risk level unconditionally to `CRITICAL`.

In [4]:
snap3 = engine.evaluate(
    asset_id='SENTINEL-2A',
    health_input=0.06,
    debris_input=0.94,
    disaster_input=0.15,
    pc=2.4e-4,
    debris_attributions=[
        Attribution(driver='pc_collision_probability', module='debris', contribution=0.68, direction='increase'),
        Attribution(driver='miss_distance_m', module='debris', contribution=0.22, direction='increase'),
    ],
    evidence={
        'secondary_id': 'FENGYUN-1C-DEB-29834',
        'miss_distance_m': 315.4,
        'time_to_tca_hours': 14.2,
    },
)
print(f'Score: {snap3.score:.1f} | Level: {snap3.level.value} | Critical Override: {snap3.critical_override_applied}')
print(f'Rationale:\n{snap3.rationale}')
print(f'Advisory Action: [{snap3.recommendations[0].action_code}] {snap3.recommendations[0].description}')

## 4. Emergency Disaster Observation Tasking (Scenario 4)
Cyclone flood event ($r_e = 0.88$, $320\text{ km}^2$) with healthy spacecraft bus ($r_h = 0.16$). Generates payload tasking recommendation.

In [5]:
snap4 = engine.evaluate(
    asset_id='SENTINEL-2A',
    health_input=0.16,
    debris_input=0.08,
    disaster_input=0.88,
    disaster_attributions=[
        Attribution(driver='flooded_area_km2', module='disaster', contribution=0.55, direction='increase'),
        Attribution(driver='population_exposure', module='disaster', contribution=0.30, direction='increase'),
    ],
    evidence={'flooded_area_km2': 320.0, 'disaster_type': 'cyclone flood'},
)
print(f'Score: {snap4.score:.1f} | Level: {snap4.level.value}')
print(f'Primary Action: [{snap4.recommendations[0].action_code}] {snap4.recommendations[0].title}')